In [0]:
from kagglehub import dataset_download
import os
import shutil

from pyspark.sql.functions import col, lit, current_timestamp

In [0]:
dbutils.widgets.text("catalog", "olist_project_dev")
dbutils.widgets.text("bronze_schema", "olist_bronze")
dbutils.widgets.text("metadata_table", "olist_source_metadata")

In [0]:
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
metadata_table_name = dbutils.widgets.get("metadata_table")

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}")

In [0]:
if not spark.catalog.tableExists(f"{catalog}.{bronze_schema}.{metadata_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{bronze_schema}.{metadata_table_name} (
            versionNum INT,
            createdTimestamp TIMESTAMP,
            updatedTimestamp TIMESTAMP,
            isActive BOOLEAN
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

metadata_df = spark.table(f"{catalog}.{bronze_schema}.{metadata_table_name}")

In [ ]:
try:
    download_path = dataset_download("olistbr/brazilian-ecommerce")
except Exception as e:
    raise RuntimeError(f"Kaggle download failed: {e}") from e

version_segment = os.path.basename(os.path.normpath(download_path))
if not version_segment.isdigit():
    raise ValueError(f"Unexpected kagglehub version segment: {version_segment!r} (path={download_path})")

num_version = int(version_segment)

if metadata_df.filter(col("versionNum") == num_version).count() > 0:
    dbutils.notebook.exit("Dataset already downloaded")

In [ ]:
volume_path = f"/Volumes/{catalog}/{bronze_schema}/raw_data"
os.makedirs(volume_path, exist_ok=True) 

for csv_file in os.listdir(download_path):
    if not csv_file.endswith(".csv"):
        continue
    shutil.copy2(f"{download_path}/{csv_file}", f"{volume_path}/{csv_file}")

In [0]:
for downloaded_file in os.listdir(volume_path):
    
    table_name = downloaded_file.replace(".csv", "").replace("_dataset", "")
    
    raw_df = spark.read.csv(f"{volume_path}/{downloaded_file}", header=True, inferSchema=False)
    
    (raw_df
     .withColumns({
         "ingestionTimestamp": current_timestamp(),
         "sourceFile": lit(downloaded_file)
     })
     .write
     .format("delta")
     .mode("overwrite")
     .saveAsTable(f"{catalog}.{bronze_schema}.{table_name}")
    )

    spark.sql(f"""
        ALTER TABLE {catalog}.{bronze_schema}.{table_name} SET TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
    """)

In [ ]:
for downloaded_file in os.listdir(download_path):
    if not downloaded_file.endswith(".csv"):
        continue

    table_name = downloaded_file.replace(".csv", "").replace("_dataset", "")

    raw_df = spark.read.csv(f"{volume_path}/{downloaded_file}", header=True, inferSchema=False)

    enriched_df = (
        raw_df.withColumns({
            "ingestionTimestamp": current_timestamp(),
            "sourceFile": lit(downloaded_file)
        })
    )

    (enriched_df
     .write
     .format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable(f"{catalog}.{bronze_schema}.{table_name}")
    )

    spark.sql(f"""
        ALTER TABLE {catalog}.{bronze_schema}.{table_name} SET TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
    """)

In [ ]:
spark.sql(f"""
    INSERT INTO {catalog}.{bronze_schema}.{metadata_table_name}
    VALUES ({num_version}, current_timestamp(), NULL, true)
""")

spark.sql(f"""
    UPDATE {catalog}.{bronze_schema}.{metadata_table_name}
    SET isActive = false, updatedTimestamp = current_timestamp()
    WHERE isActive = true AND versionNum <> {num_version}
""")